# Does longer context help a small character-level transformer?

Daniel Moody (23370157) & Nandakishore Vinayakrishnan (23070854)

**Research question:** with training characters held fixed, does increasing context length from 32 to 128 to 256 lower test bits-per-character for a 4-layer character transformer on TinyShakespeare?

Baseline-gate pipeline: data, models, metric, training loop and both reference conditions. Use a T4 GPU runtime. The test split is not used here; it is scored once, after every run and decision is frozen.

## Layer 1 of 4 - Data preparation

Run this notebook first. It downloads TinyShakespeare when needed, makes the contiguous line-aligned 80/10/10 split, and writes the split text files and vocabulary summary used by the later layers.


In [ ]:
import json
import urllib.request
from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (current, *current.parents) if (path / ".git").exists()), current)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RUNS_DIR = RESULTS_DIR / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
for directory in (DATA_DIR, RESULTS_DIR, RUNS_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = DATA_DIR / "input.txt"


## 1. Data

TinyShakespeare is downloaded from Karpathy's char-rnn repository (MIT licence; the plays are public domain) and split contiguously 80/10/10. Each cut is moved forward to the next line break, so no line is split across sets. The vocabulary comes from the training split only, and any validation or test characters missing from it are reported.

In [ ]:
if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

text = DATA_PATH.read_text(encoding="utf-8")

def next_line_start(position):
    return text.index("\n", position) + 1

train_end = next_line_start(int(0.8 * len(text)))
val_end = next_line_start(int(0.9 * len(text)))
train_text, val_text, test_text = text[:train_end], text[train_end:val_end], text[val_end:]

vocab = sorted(set(train_text))
char_to_id = {char: i for i, char in enumerate(vocab)}
unseen_chars = {name: sorted(set(split) - set(vocab)) for name, split in [("val", val_text), ("test", test_text)]}

VOCAB_SIZE = len(vocab)

data_summary = {
    "total_chars": len(text),
    "split_chars": {"train": len(train_text), "val": len(val_text), "test": len(test_text)},
    "vocab_size": len(vocab),
    "unseen_chars": unseen_chars,
}
(RESULTS_DIR / "data_summary.json").write_text(json.dumps(data_summary, indent=2))
for split_name, split_text in (("train", train_text), ("val", val_text), ("test", test_text)):
    (DATA_DIR / f"{split_name}.txt").write_text(split_text, encoding="utf-8")
data_summary
